# 03 — Intent Classification & Evaluation

Train real classifiers on NOVA's intent dataset and learn to **read the
metrics that matter**: precision, recall, F1, macro F1, confusion matrices.

Production use: `backend/app/nlp/train_intent_model.py` (this notebook is the
interactive counterpart — same concepts, smaller experiments).
Docs: `docs/nlp/03_classification.md` · Report: `docs/reports/intent_model.md`

In [ ]:
import math, re, sys
from pathlib import Path

cwd = Path.cwd()
backend_dir = cwd.parent / "backend" if cwd.name == "notebooks" else cwd / "backend"
sys.path.insert(0, str(backend_dir))

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print("imports OK")
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score, recall_score)
from app.nlp.intent import DATASET_PATH, classify_intent, load_dataset

## 1. The dataset

12 intents, ~489 hand-written utterances. Real-world-ish: typos, colloquial
phrases, and *inherent ambiguities* included on purpose.

In [ ]:
ds = load_dataset()
rows = [{"intent": i["tag"], "patterns": len(i["patterns"])} for i in ds.intents]
df = pd.DataFrame(rows).set_index("intent").sort_values("patterns", ascending=False)
print(df.to_string())
print("total:", df.patterns.sum())

## 2. Train/test split — and why we prefer cross-validation

Hold-out is fine for big data. With ~400 samples, *which* 20% lands in the
test set changes macro F1 by ±0.08 — the number lies. **Stratified k-fold CV:**
k=5 stratified folds; every sample is tested exactly once; report mean ± std.

In [ ]:
ds = load_dataset()
X = [p for i in ds.intents for p in i["patterns"]]
y = [i["tag"] for i in ds.intents for p in i["patterns"]]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def make(clf):
    # stop_words=None on purpose — who/what/how distinguish intents!
    return Pipeline([("tfidf", TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2))),
                     ("clf", clf)])

for name, clf in [("LogReg", LogisticRegression(max_iter=2000)),
                  ("MNB", MultinomialNB(alpha=0.1)),
                  ("LinearSVC", LinearSVC())]:
    scores = cross_val_score(make(clf), X, y, cv=5, scoring="f1_macro")
    print(f"{name:10} CV macro-F1 = {scores.mean():.3f} ± {scores.std():.3f}")

## 3. The metrics, defined properly

For one class (one-vs-rest thinking):

```
precision = TP / (TP + FP)     "when predicted X, correct how often?"
recall    = TP / (TP + FN)     "of real X, how many caught?"
F1        = 2·P·R / (P + R)    harmonic mean — low if either is low
macro F1  = mean of F1 over all classes (each class votes equally)
```

Accuracy hides class imbalance: a model that always predicts the majority
class can be 40% accurate and still useless. Macro F1 cannot cheat that way.

In [ ]:
pipe = make(LinearSVC()).fit(Xtr, ytr)
preds = pipe.predict(Xte)

print(f"accuracy     : {accuracy_score(yte, preds):.3f}")
print(f"precision(m) : {precision_score(yte, preds, average='macro', zero_division=0):.3f}")
print(f"recall(m)    : {recall_score(yte, preds, average='macro', zero_division=0):.3f}")
print(f"F1 (macro)   : {f1_score(yte, preds, average='macro', zero_division=0):.3f}")
print()
print(classification_report(yte, preds, zero_division=0))

## 4. Reading a confusion matrix

Rows = actual, columns = predicted. Diagonal = correct. Off-diagonal cells
**name** the failure: e.g. identity → capabilities means the model saw
"who are you" features inside the capabilities region.

In [ ]:
labels = sorted(set(y))
cm = confusion_matrix(yte, preds, labels=labels)
cm_df = pd.DataFrame(cm, index=[l[:12] for l in labels], columns=[l[:12] for l in labels])
cm_df.style.background_gradient(cmap="Blues").format("{:.0f}")

## 5. Low-confidence fallback (the `unknown` intent)

A deployed classifier must say **"I don't know"** instead of guessing.
Calibrated models give probabilities; we threshold them (0.35).

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
cal = Pipeline([("tfidf", TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2))),
                ("clf", CalibratedClassifierCV(LinearSVC(), cv=3))]).fit(Xtr, ytr)

proba = dict(zip(cal.classes_, cal.predict_proba(["zxcvbn mnbvc qwe"])[0]))
top = max(proba, key=proba.get)
print("top class:", top, "conf:", round(proba[top], 3))
print("-> unknown" if (top != "unknown" and proba[top] < 0.35) else "-> accepted")

## 6. Production behavior

The saved model (Linear SVM calibrated, macro F1 ≈ 0.76 CV) lives in
`backend/artifacts/intent/`. Serving adds: lazy loading, artifact-missing
fallbacks (TF-IDF baseline → keyword rules), and empty-input handling.

In [ ]:
for msg in ["hey, what's up?", "I can't remember my login password",
             "what time is it", "qwerty zxcvbn mnbvc"]:
    r = classify_intent(msg)
    print(f"{msg!r:45} -> {r.label:16} {r.confidence:.2f}")